# dataset_labeling_2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances


In [1]:
import pandas as pd

In [2]:
def assign_destination(destination: str) -> int:
    if destination == "Broadcast": 
        return 1
    else:
        return 0


def check_null_value(string_to_check: str) -> str:
    if pd.isna(string_to_check):
        return ""
    else:
        return string_to_check


def assign_device(device: str, source: str) -> str:
    if device in ["Phone", "Phone Scan", "Phone Google"]:
        return "Phone"
    else:
        return device


def encode_company_id(company: str, source: str, service_data: str) -> str:
    # Defined, but not named company identifiers f.e. 0x34f5
    if str(company).startswith("0x") and len(company) == 6:
        return "Unknown"
    # Assign the Galaxy S22 "Anonymous" source the company Samsung
    elif pd.isna(company) and source == "Anonymous":
        return "Samsung Electronics Co. Ltd."
    # Reason?
    elif pd.isna(company) and str(service_data).startswith("4a17235"):
        return "Samsung Electronics Co. Ltd."
    elif pd.isna(company):
        return "Undefined"
    else:
        return company


def extract_time(delta_string: str) -> int:
    if pd.isna(delta_string):
        return 0
    else:
        s = str(delta_string)
        num = "".join(ch for ch in s if ch.isdigit())
        return int(num) if num else 0


def check_company_id_existence(company_id: str) -> int:
    if company_id == "Undefined":
        return 0
    else:
        return 1


def extract_length(entry_to_exctract: str) -> int:
    if pd.isna(entry_to_exctract):
        return 0
    else:
        return len(entry_to_exctract)


def is_adv_channel(channel: int) -> bool:
    if channel in [37, 38, 39]:
        return True
    else:
        return False


def check_entry_existence(entry_to_check: str) -> int:
    if pd.isna(entry_to_check):
        return 0
    else:
        return 1


In [3]:
# Read the input CSV file and rename some columns - robust to both naming schemes (Channel vs Channel_Index, UUID16 vs UUID_16)
def read_data_to_df(file_name: str) -> pd.DataFrame:
    dataset_df = pd.read_csv(file_name, encoding='ISO-8859-1', low_memory=False)
    dataset_df.columns = dataset_df.columns.str.replace("Packet time (start to end)", "packet_start_end")
    dataset_df.columns = dataset_df.columns.str.replace("Delta time (end to start)", "delta_end_start")
    dataset_df.columns = dataset_df.columns.str.replace("Delta time (start to start)", "delta_start_start")
    dataset_df.columns = dataset_df.columns.str.replace(" ", "_")
    dataset_df.columns = dataset_df.columns.str.replace("Test", "Device")
    if "Channel" in dataset_df.columns and "Channel_Index" not in dataset_df.columns:
        dataset_df = dataset_df.rename(columns={"Channel": "Channel_Index"})
    if "UUID16" in dataset_df.columns and "UUID_16" not in dataset_df.columns:
        dataset_df = dataset_df.rename(columns={"UUID16": "UUID_16"})
    if "AD_Type" in dataset_df.columns and "Type" not in dataset_df.columns:
        dataset_df = dataset_df.rename(columns={"AD_Type": "Type"})
    return dataset_df

# --- 10h Seewis: iPhone / AirPods / Samsung SmartTag, two groups at different distances ---
PHONE_MAP = {
 "2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours0To2.csv": (['44:55:3c:29:d4:d8', '48:29:85:2c:a6:87', '4d:39:86:43:78:78', '55:38:ef:cf:40:14', '5a:9e:c1:f9:4f:81', '5e:61:60:ab:23:36', '6b:eb:0f:72:4d:54', '6c:38:7c:46:cc:33', '6c:d9:d6:e7:5a:20'], ['43:65:e1:d6:f2:eb', '4a:9b:a5:44:83:28', '51:dc:9f:2e:24:8c', '5e:11:30:21:6c:39', '65:53:f9:ec:76:2e', '6c:8a:13:1a:55:f5', '6d:d3:53:77:f9:6f', '7b:0c:20:7c:f3:7e', '7d:89:59:c3:7e:eb']),
 "2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours2To4.csv": (['46:96:8f:2f:d9:8a', '48:11:9c:74:ec:d5', '48:30:08:42:25:84', '4d:39:86:43:78:78', '4f:be:34:70:ff:75', '57:ce:dd:c0:b6:26', '65:d9:60:e6:7f:29', '69:78:59:e7:d2:47', '6e:63:e4:c3:07:60', '71:92:07:02:71:63'], ['45:4a:a5:04:0c:b0', '55:72:05:e9:df:2e', '5c:91:59:7b:db:23', '64:b5:80:7f:b8:fa', '6d:d3:53:77:f9:6f', '77:02:09:4b:c5:66', '77:f1:93:e1:c9:72', '78:a6:76:b2:e5:aa', '7c:cf:f4:25:da:3f', '7e:38:23:d4:52:8c']),
 "2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours4To6.csv": (['5d:19:fd:25:b8:48', '60:65:6c:80:36:53', '63:a2:1f:b3:59:53', '65:04:af:98:44:38', '70:bb:de:a0:bf:bc', '71:92:07:02:71:63', '71:dd:54:cc:ee:d1', '72:8a:20:76:97:b4', '7e:4e:3a:10:74:30'], ['45:06:73:c7:aa:6b', '47:0c:86:14:fc:35', '5e:41:5a:e8:2a:95', '63:06:e8:f4:a8:9d', '64:b5:80:7f:b8:fa', '6e:48:38:bc:79:8e', '70:43:34:3e:97:9c', '75:fb:9a:e0:3a:dc', '77:71:bd:69:95:63']),
 "2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours6To8.csv": (['4f:d6:fb:c4:bb:17', '51:25:e8:ba:d0:37', '53:78:2a:99:51:c4', '5a:6f:a3:c4:73:9d', '5b:a2:77:ee:d2:ba', '5d:19:fd:25:b8:48', '61:81:87:7e:ea:1d', '66:21:70:1b:ea:97', '7d:11:82:96:70:6e'], ['51:3a:66:74:75:56', '53:90:80:a3:a5:c2', '53:cb:b5:33:83:1b', '56:e1:2a:dc:8f:db', '58:0c:89:9c:47:13', '5e:41:5a:e8:2a:95', '61:6f:d6:d8:aa:15', '6b:1b:a7:ad:37:82', '6b:7d:64:cc:89:65', '7f:b8:bc:a7:b0:d0']),
 "2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours8To10.csv": (['46:19:70:e0:44:a0', '49:ed:ba:13:2c:ad', '4f:60:3b:29:03:42', '53:78:2a:99:51:c4', '6f:43:c3:80:8e:38', '71:64:ca:0c:5c:c3', '7e:2b:e1:81:3c:a9', '7f:8c:bf:df:d7:fd'], ['43:7a:c2:cc:5b:85', '49:aa:2b:12:23:0a', '4d:98:33:f9:92:58', '4e:d7:bd:a4:1c:20', '5b:93:18:64:f4:6c', '60:65:c9:f2:37:c6', '6e:f8:23:43:b8:79', '71:2e:ce:1b:1c:04', '7f:b8:bc:a7:b0:d0']),
}

AIRPODS_MAP = {
 "2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours0To2.csv": (['45:1e:f4:84:e8:9c', '49:87:be:bd:07:52', '5a:40:26:06:86:cc', '5f:d2:78:68:13:6e', '62:fb:d6:f2:32:ad', '67:90:72:12:d8:fc', '6c:f7:8e:1f:e7:1b', '7a:93:41:0c:cd:23'], ['49:51:bd:40:bb:5f', '4b:4a:73:0c:e0:7d', '4d:1c:6d:88:eb:8b', '4f:a5:b0:a3:5e:6d', '4f:da:4a:07:b3:8b', '5c:c6:6b:62:c1:bb', '60:1d:56:bf:20:52', '6a:89:6f:6d:73:6c', '71:6a:3f:bd:57:32']),
 "2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours2To4.csv": (['4a:40:94:86:7d:4e', '4e:de:79:9c:44:f0', '55:2e:ed:be:9a:72', '56:b4:4a:6a:95:e2', '5c:55:68:5a:da:04', '61:48:89:96:0c:76', '62:26:16:81:ca:b0', '6d:56:f6:60:d5:3d', '73:6c:0d:57:63:54'], ['44:13:57:14:db:71', '4b:79:99:74:8e:6b', '5a:08:c3:7c:35:17', '5c:c6:6b:62:c1:bb', '5f:e0:ee:90:ce:81', '60:8b:26:24:5b:69', '6a:69:d1:47:bc:e3', '79:b1:92:45:d2:5f', '7f:40:6d:2b:7e:c5']),
 "2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours4To6.csv": (['43:ff:5a:27:11:d6', '50:12:f3:ad:6b:a3', '50:47:fc:8a:22:4b', '55:2e:ed:be:9a:72', '56:32:5d:62:1b:51', '69:d4:1e:de:10:0d', '6c:33:1a:74:b1:23', '72:25:c5:f2:6f:09'], ['40:4a:6a:63:ab:6d', '4a:de:37:1c:15:f7', '4e:d7:9d:1e:b4:07', '52:90:c2:28:4f:56', '6d:fe:63:a2:1e:8c', '77:b2:69:fe:2f:bf', '77:b7:e8:13:ee:93', '7c:23:dd:c1:ea:e4', '7f:40:6d:2b:7e:c5']),
 "2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours6To8.csv": (['49:62:c8:fd:e7:05', '4d:54:d3:d2:eb:66', '58:c8:ce:b3:79:95', '5d:3f:c5:7c:c9:ab', '5e:02:8e:55:ba:7a', '5e:06:85:6c:5c:d1', '5e:a9:b9:53:16:0f', '61:53:a0:5f:60:83', '6f:c9:6b:8d:90:41'], ['47:ba:67:31:5b:25', '54:92:02:f1:d6:79', '58:da:5e:94:89:da', '68:f4:61:da:39:c5', '6d:ed:6a:67:ac:82', '6d:fe:63:a2:1e:8c', '72:5b:5c:8a:37:c6', '75:11:02:72:c5:e8', '7a:e9:a8:f4:bf:2f']),
 "2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours8To10.csv": (['43:e4:87:ee:09:9c', '47:9f:72:17:84:0c', '5e:3b:00:c1:ac:cd', '68:2e:46:84:58:d3', '6d:61:b8:20:c1:78', '6f:c9:6b:8d:90:41', '71:a0:8f:16:55:de', '7b:7d:5e:ce:71:58', '7c:5d:d5:91:af:19'], ['43:7a:b2:d9:71:e6', '45:6c:7d:eb:b3:79', '4a:c5:9e:25:4c:c2', '4c:79:c5:b2:12:ae', '51:2c:ae:30:bd:a0', '54:48:86:38:b3:2c', '6d:ed:6a:67:ac:82', '79:01:82:b0:36:fc', '7f:7f:d2:43:81:b5']),
}

SMARTTAG_MAP = {
 "2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours0To2.csv": (['43:79:31:f0:dd:e5', '4b:b7:f4:16:3e:cd', '4e:00:b4:e2:de:54', '4e:40:b2:65:2a:66', '51:2a:a2:db:47:c2', '54:52:38:a3:4a:7a', '54:ba:d4:3b:45:34', '58:ff:96:58:ad:c4', '60:c8:b4:5d:8c:19', '63:bb:11:9b:a1:68', '6e:a5:bf:6c:66:d3', '74:ed:a4:8b:1f:7a', '75:22:c5:c9:8f:70', '77:c0:16:1c:13:2a', '79:2c:51:0b:03:92', '7c:b3:f5:e1:36:d4'], ['41:e9:d9:4c:88:0d', '43:15:36:79:8f:ad', '49:54:13:b4:7f:68', '5e:88:79:90:60:73', '5f:16:b8:bd:1c:ac', '60:1c:ff:e4:1d:cb', '60:5c:26:16:cb:67', '63:3a:b6:8c:7b:a4', '64:d5:07:4e:d7:ca', '68:97:7d:42:8c:9a', '6b:1f:9e:99:fd:98', '6d:ae:85:45:6e:27', '6f:64:41:2c:d0:f1', '70:c8:18:bb:23:57', '71:3c:f3:3b:3f:f3', '73:80:09:ca:22:de']),
 "2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours2To4.csv": (['40:d7:ef:25:ae:eb', '47:89:c1:64:1d:e5', '4d:4f:b8:ff:f7:22', '55:79:f8:3a:b2:d3', '56:5f:ef:04:1a:06', '59:59:3f:34:3e:7c', '60:ff:4a:e0:94:14', '64:7d:cf:23:fe:3e', '65:25:93:9a:25:05', '66:de:8a:51:1a:82', '68:23:7e:71:19:30', '6a:25:97:4b:01:c3', '6c:7d:3b:7e:b4:1b', '70:32:17:a4:85:37', '73:6a:4b:57:12:e0', '77:8e:20:dd:a8:d0'], ['4e:68:55:85:09:f1', '4e:7d:65:d7:5f:9d', '4f:b3:28:7e:62:b6', '51:f9:15:60:96:0e', '54:a3:75:3b:fe:7d', '57:a2:d7:58:a0:2a', '5f:f8:9f:0c:0f:c8', '66:9b:19:cf:03:2d', '66:d0:2d:08:8d:4a', '68:8d:38:c2:00:4f', '6f:4e:db:8c:23:d9', '76:41:87:a1:1c:a0', '76:75:d9:09:c6:a4', '76:91:97:28:cb:b0', '7d:53:be:aa:68:23']),
 "2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours4To6.csv": (['44:01:01:37:ee:0f', '49:12:9d:80:51:9d', '4a:78:2c:c2:24:d8', '4b:58:37:07:ef:fa', '5f:bc:c4:fd:e9:00', '61:52:51:e8:10:ee', '62:3f:72:dc:39:b0', '68:0c:18:6e:c6:6a', '73:56:0e:54:db:02'], ['46:70:ba:d6:6d:2b', '47:ca:72:d7:a8:60', '4f:e9:5a:67:6a:4f', '55:0d:10:d3:57:47', '5a:ae:c4:75:b8:34', '66:c7:17:96:96:f5', '69:75:92:85:e8:2a', '7b:1e:54:77:d1:ae']),
 "2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours6To8.csv": (['42:bb:ab:0d:75:05', '4b:8e:e8:c3:c8:44', '54:17:13:47:af:c7', '57:fd:c6:4a:18:51', '5b:0e:ad:eb:0c:33', '69:3d:29:30:96:2e', '74:44:64:f3:84:db', '74:b2:e7:4b:1e:a5'], ['40:50:4a:cb:ca:a1', '42:5e:19:78:7c:fb', '4d:fd:60:ef:cc:b5', '5c:41:d7:62:fc:25', '69:f1:25:04:4e:55', '6a:4c:7e:8e:00:4f', '73:d6:f1:01:fa:4d', '74:ba:ab:5e:cd:01']),
 "2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours8To10.csv": (['46:e3:f1:80:ff:62', '48:62:6d:44:c3:a7', '49:9a:95:ac:c9:89', '4e:2b:67:37:48:9f', '4e:4a:ad:31:f9:f9', '56:80:ff:b9:b3:fc', '56:c3:b4:b6:3d:4e', '5e:0e:87:07:da:40', '71:93:15:05:3e:ed', '76:1f:d7:4a:bd:a8', '77:b8:ea:6c:2a:b2', '78:00:4f:dd:57:b1'], ['4c:58:b3:81:87:d3', '4e:90:1a:13:aa:88', '50:da:d3:37:f9:01', '53:07:3a:ae:d0:cd', '54:14:98:ac:2e:c2', '57:52:d1:af:e7:10', '5a:41:d0:c4:74:d8', '5c:77:d4:91:d6:66', '64:29:cf:4c:e1:fe', '64:44:39:99:da:82', '6c:18:8d:8c:6d:b7', '74:21:b8:84:4a:41', '7d:2b:eb:55:97:e5', '7e:7d:1a:c1:b9:40']),
}

BUDS_G1 = []
BUDS_G2 = []
AIRTAG_G1 = []
AIRTAG_G2 = []

def build_source_map(phone_g1, phone_g2, airpods_g1=None, airpods_g2=None, smarttag_g1=None, smarttag_g2=None):
    ag1 = airpods_g1 if airpods_g1 is not None else []
    ag2 = airpods_g2 if airpods_g2 is not None else []
    sg1 = smarttag_g1 if smarttag_g1 is not None else []
    sg2 = smarttag_g2 if smarttag_g2 is not None else []
    m={}
    for s in phone_g1: m[s]=("Phone","Phone Scan",1)
    for s in phone_g2: m[s]=("Phone","Phone Scan",2)
    for s in ag1: m[s]=("Buds","Buds",1)
    for s in ag2: m[s]=("Buds","Buds",2)
    for s in sg1: m[s]=("Smart Tag","Smart Tag",1)
    for s in sg2: m[s]=("Smart Tag","Smart Tag",2)
    return m

def initialize_source_dictionaries(all_unique_sources: list) -> None:
    for source in all_unique_sources:
        if str(source) != "nan":
            source_dictionaries[source] = {}
            source_dictionaries[source]["count"] = 0
            source_dictionaries[source]["malformed_count"] = 0
            source_dictionaries[source]["highest_rssi"] = 0
            source_dictionaries[source]["lowest_rssi"] = -100
            source_dictionaries[source]["average_rssi"] = 0
            source_dictionaries[source]["first_occurence"] = -1
            source_dictionaries[source]["last_occurence"] = 0
            source_dictionaries[source]["device"] = "empty"
            source_dictionaries[source]["sub_device"] = ""
            source_dictionaries[source]["group_id"] = 0


In [4]:
# Assign values for each row with either simple transformations or direct value usage
def fill_labelled_columns(packet_data: tuple, device: str, sub_device: str, group_id: int):

    row_data = {
        "time": packet_data.Time,
        "source": packet_data.Source,
        "destination": packet_data.Destination,
        "is_broadcast": assign_destination(packet_data.Destination),
        "length": packet_data.Length,
        "info": packet_data.Info,
        "rssi": packet_data.RSSI,
        "company_id": encode_company_id(packet_data.Company_ID, packet_data.Source, packet_data.Service_Data),
        "has_company_id": check_company_id_existence(encode_company_id(packet_data.Company_ID, packet_data.Source, packet_data.Service_Data)),
        "channel": packet_data.Channel_Index,
        "is_adv_channel": is_adv_channel(packet_data.Channel_Index),
        "device_name": check_null_value(packet_data.Device_Name),
        "uuid16": check_null_value(packet_data.UUID_16),
        "has_uuid16": check_entry_existence(packet_data.UUID_16),
        "len_uuid16": extract_length(packet_data.UUID_16),
        "uuid128": check_null_value(packet_data.UUID128),
        "has_uuid128": check_entry_existence(packet_data.UUID128),
        "data": check_null_value(packet_data.Data),
        "len_data": extract_length(packet_data.Data),
        "ad_type": check_null_value(packet_data.Type),
        "len_ad_type": extract_length(packet_data.Data),
        "service_data": check_null_value(packet_data.Service_Data),
        "len_service_data": extract_length(packet_data.Service_Data),
        "crc": check_null_value(packet_data.CRC),
        "labelled_device": assign_device(device, packet_data.Source),
        "sublabel_device": sub_device,
        "group_id": group_id,
        "group_label_bin": 1 if group_id != 0 else 0,
        "group_label_multi": group_id,  # 0=noise, 1=G1, 2=G2
        "time_start_end": extract_time(packet_data.packet_start_end),
        "delta_end_start": extract_time(packet_data.delta_end_start),
        "delta_start_start": extract_time(packet_data.delta_start_start)
    }

    # To ensure labeling quality, phone-based labels which do not a valid company_id are labeled empty
    # Valid company_ids include: ["Samsung Electronics Co. Ltd.", "Undefined", "Unknown"]
    if row_data["labelled_device"] in ['Phone'] and row_data["company_id"] not in phone_companies:
        row_data["labelled_device"] = "empty"
        row_data["sublabel_device"] = "empty"
        row_data["group_id"] = 0
        row_data["group_label_bin"] = 0
        row_data["group_label_multi"] = 0

    return row_data


In [5]:
# Main raw dataset input parsing method
def parse_dataframe(input_dataframe: pd.DataFrame):
    nan_discarded_counter = 0
    malformed_discarded_counter = 0
    smart_tag_list = []
    phone_list = []
    buds_list = []

    dict_mean_rssi = (
        pd.to_numeric(
            input_dataframe["RSSI"].astype(str).str.replace("dBm", "", regex=False).str.strip(),
            errors="coerce"
        )
        .groupby(input_dataframe["Source"])
        .mean()
        .to_dict()
    )

    for row in input_dataframe.itertuples(index=False):
        if str(row.Source) == "nan":
            nan_discarded_counter += 1
            continue

        source_dictionaries[row.Source]["count"] += 1

        if "Malformed Packet" in row.Info:
            source_dictionaries[row.Source]["malformed_count"] += 1
            malformed_discarded_counter += 1
            continue

        # Parse RSSI string "-50 dBm" -> int
        try:
            current_rssi = int(str(row.RSSI).replace("dBm","").strip())
        except:
            current_rssi = -100
        if source_dictionaries[row.Source]["first_occurence"] == -1:
            source_dictionaries[row.Source]["first_occurence"] = row.Time
        if row.Time > source_dictionaries[row.Source]["last_occurence"]:
            source_dictionaries[row.Source]["last_occurence"] = row.Time
        if current_rssi > source_dictionaries[row.Source]["lowest_rssi"]:
            source_dictionaries[row.Source]["lowest_rssi"] = current_rssi
        if current_rssi < source_dictionaries[row.Source]["highest_rssi"]:
            source_dictionaries[row.Source]["highest_rssi"] = current_rssi
        if source_dictionaries[row.Source]["average_rssi"] == 0:
            source_dictionaries[row.Source]["average_rssi"] = dict_mean_rssi[row.Source]
            
        # --- Priority 1: Whitelist / ground truth MAP for multi-group ---
        # This overrides heuristics and ensures G1 vs G2 separation even with identical packet content / same RSSI
        if row.Source in SOURCE_TO_GROUP:
            dev, sub, grp = SOURCE_TO_GROUP[row.Source]
            source_dictionaries[row.Source]["device"] = dev
            source_dictionaries[row.Source]["sub_device"] = sub
            source_dictionaries[row.Source]["group_id"] = grp
        else:
            # --- Fallback heuristics for non-target / noise and for completeness ---
            # Smart Tag fallback disabled for 2Groups hybrid dataset (no Smart Tags in experiment)
            # if row.Device_Name == "Smart Tag":
            #     source_dictionaries[row.Source]["device"] = "Smart Tag"
            #     source_dictionaries[row.Source]["sub_device"] = "Smart Tag"
            # if row.UUID_16 == "Samsung Electronics Co., Ltd.,Samsung Electronics Co., Ltd.":
            #     source_dictionaries[row.Source]["device"] = "Smart Tag"
            #     source_dictionaries[row.Source]["sub_device"] = "Smart Tag"
            pass
            # Phone via UUID heuristics
            if row.UUID_16 == "Google LLC" and (row.Length == 63 or row.Length == 42 or row.Length == 120): 
                if source_dictionaries[row.Source]["device"] == "empty":
                    source_dictionaries[row.Source]["device"] = "Phone"
                    source_dictionaries[row.Source]["sub_device"] = "Phone Google"
            if row.Info == "ADV_EXT_IND" and row.Length == 39:
                if source_dictionaries[row.Source]["device"] == "empty":
                    source_dictionaries[row.Source]["device"] = "Phone"
                    source_dictionaries[row.Source]["sub_device"] = "Phone Phone"
            if row.UUID_16 == "Samsung Electronics Co. Ltd." and row.Length == 57: 
                if source_dictionaries[row.Source]["device"] == "empty":
                    source_dictionaries[row.Source]["device"] = "Phone"
                    source_dictionaries[row.Source]["sub_device"] = "Phone Phone"
            if row.Info == "SCAN_REQ" and row.Length == 38 and row.Destination in target_group_rsp_scrs:
                if source_dictionaries[row.Source]["device"] == "empty":
                    source_dictionaries[row.Source]["device"] = "Phone"
                    source_dictionaries[row.Source]["sub_device"] = "Phone Scan"
            # Buds fallback: any Samsung ADV_SCAN_IND L63 with Samsung company is Buds (covers unknown Buds)
            # But whitelist already covers G1/G2 Buds, so here we only label remaining Buds as noise or generic
            if row.Info == "ADV_SCAN_IND" and row.Length == 63 and row.Company_ID == "Samsung Electronics Co. Ltd.":
                if source_dictionaries[row.Source]["device"] == "empty" and row.Source not in ALL_BUDS:
                    # keep as empty to avoid mislabeling interferers; or label generic Buds with group 0
                    pass
            # AirTag fallback: Apple ADV_IND L63 (all AirTags) - whitelist covers G1/G2, rest treated as generic AirTag noise
            if row.Company_ID == "Apple, Inc." and row.Info == "ADV_IND" and row.Length == 63:
                if source_dictionaries[row.Source]["device"] == "empty" and row.Source not in ALL_AIRTAGS:
                    # Keep as empty/noise for modeling; if you want to keep as noise AirTag, uncomment:
                    # source_dictionaries[row.Source]["device"] = "AirTag"
                    # source_dictionaries[row.Source]["sub_device"] = "AirTag"
                    # source_dictionaries[row.Source]["group_id"] = 0
                    pass

        # After assigning all values, create a transformed row for the final list of rows
        row_data = fill_labelled_columns(row, source_dictionaries[row.Source]["device"], source_dictionaries[row.Source]["sub_device"], source_dictionaries[row.Source]["group_id"])
        final_list.append(row_data)

    return nan_discarded_counter, malformed_discarded_counter


In [6]:
def find_empty_and_malformed_sources():
    only_malformed_sources = []
    empty_sources = []

    for key in source_dictionaries.keys():
        count = source_dictionaries[key]["count"]
        malformed_count = source_dictionaries[key]["malformed_count"]
        device = source_dictionaries[key]["device"]

        if (count - malformed_count) == 0:
            only_malformed_sources.append(key)
        
        if (count - malformed_count) != 0 and device == "empty":
            empty_sources.append(key)

    print(f"Only Malformed Sources: {len(only_malformed_sources)}")
    print(f"Empty Sources: {len(empty_sources)}")

    return only_malformed_sources, empty_sources

def reassign_sources():
    empty_occurrences = 0
    new_final_list = []

    for row_dict in final_list:
        if row_dict["labelled_device"] == "empty" and row_dict["company_id"] in phone_companies:
            current_source = row_dict["source"]
            # Only reassign if source has known device in dictionary (prevents overriding whitelist)
            if source_dictionaries[current_source]["device"] != "empty":
                row_dict["labelled_device"] = source_dictionaries[current_source]["device"]
                row_dict["sublabel_device"] = source_dictionaries[current_source]["sub_device"]
                row_dict["group_id"] = source_dictionaries[current_source]["group_id"]
                row_dict["group_label_bin"] = 1 if source_dictionaries[current_source]["group_id"] !=0 else 0
                row_dict["group_label_multi"] = source_dictionaries[current_source]["group_id"]

        if row_dict["labelled_device"] != "empty":
            new_final_list.append(row_dict)
        else:
            new_final_list.append(row_dict)
            empty_occurrences += 1

    return empty_occurrences, new_final_list


In [7]:
# Create the final dataframe based on the final_list rows
def create_labelled_dataframe(entry_list: list):
    new_df = pd.DataFrame(data=entry_list, columns=list((final_list[0].keys())))        
    return new_df

def write_new_dataframe(file_to_write: str, df_to_write: pd.DataFrame):
    df_to_write.to_csv(file_to_write, encoding='utf-8', index=False)
    print(f"New CSV written as: {file_to_write}")

def info_extractor(df_input: pd.DataFrame, wanted_label: str):
    seen_sources = {}
    for row in df_input.itertuples(index=False):
        if row.source not in seen_sources.keys() and row.sublabel_device == wanted_label:
            seen_sources[row.source] = {}
            seen_sources[row.source]["first"] = row.time
            seen_sources[row.source]["last"] = row.time
            seen_sources[row.source]["count"] = source_dictionaries[row.source]["count"]
            continue
        if row.source in seen_sources.keys() and row.sublabel_device == wanted_label:
            if seen_sources[row.source]["last"] < row.time:
                seen_sources[row.source]["last"] = row.time
    return seen_sources

def assign_majority_label(input_df: pd.DataFrame):
    # Compute the majority label per source for labelled_device, sublabel and group
    majority_labels = input_df.groupby('source')['labelled_device'].agg(lambda x: x.mode()[0])
    majority_sub = input_df.groupby('source')['sublabel_device'].agg(lambda x: x.mode()[0])
    majority_group = input_df.groupby('source')['group_id'].agg(lambda x: x.mode()[0])
    input_df['labelled_device'] = input_df['source'].map(majority_labels)
    input_df['sublabel_device'] = input_df['source'].map(majority_sub)
    input_df['group_id'] = input_df['source'].map(majority_group)
    input_df['group_label_bin'] = (input_df['group_id'] != 0).astype(int)
    input_df['group_label_multi'] = input_df['group_id']
    return input_df


In [8]:
##############################
#        CODE FIELD 0        #
##############################

# Global variables
final_list = []
source_dictionaries = {}
phone_companies = ["Apple, Inc.", "Undefined", "Unknown"]
undefined_list = []


In [9]:
##############################
#        CODE FIELD 1 - loop 5x2h blocks        #
##############################

import pathlib
base = pathlib.Path("../../data/2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances")
files = sorted([f for f in list(base.glob("2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours*.csv")) if "labeled" not in f.name])
print([f.name for f in files])

for path_to_file in files:
    # reset per-file globals
    final_list = []
    source_dictionaries = {}
    fname = pathlib.Path(path_to_file).name
    phone_g1, phone_g2 = PHONE_MAP[fname]
    airpods_g1, airpods_g2 = AIRPODS_MAP[fname]
    smarttag_g1, smarttag_g2 = SMARTTAG_MAP[fname]
    SOURCE_TO_GROUP = build_source_map(phone_g1, phone_g2, airpods_g1, airpods_g2, smarttag_g1, smarttag_g2)
    ALL_BUDS = set(airpods_g1 + airpods_g2)
    ALL_SMARTTAGS = set(smarttag_g1 + smarttag_g2)
    ALL_AIRTAGS = set()
    ALL_BUDS = set(airpods_g1 + airpods_g2)
    ALL_PHONES = set(phone_g1 + phone_g2)

    raw_dataframe = read_data_to_df(str(path_to_file))
    all_sources = raw_dataframe["Source"].unique()
    target_group_rsp_scrs = raw_dataframe.loc[(raw_dataframe["Info"] == "SCAN_RSP") & (raw_dataframe["Device_Name"] == "AirPods"), "Source"].unique()
    if len(target_group_rsp_scrs)==0:
        target_group_rsp_scrs = phone_g1 + phone_g2
    print(f"\\n=== {fname} ===")
    print(f"target_group_rsp_scrs: {list(target_group_rsp_scrs)}")
    initialize_source_dictionaries(all_sources)
    nan_counter, malformed_counter = parse_dataframe(raw_dataframe)
    malformed_list, empty_list = find_empty_and_malformed_sources()
    empty_counter, filtered_list = reassign_sources()
    labelled_dataframe = create_labelled_dataframe(filtered_list)
    relevant_sources = labelled_dataframe["source"].unique()
    print(f"Sources total: {len(all_sources)}, labelled rows: {len(labelled_dataframe)}")

    # CODE FIELD 2 inline (no RSSI filtering, whitelist only)
    whitelisted_sources = set(SOURCE_TO_GROUP.keys())
    print("--------------------------------------------------")
    print("FINAL DATAFRAME (before majority vote)")
    print(f"Length Raw: {len(raw_dataframe)}, Labelled: {len(labelled_dataframe)}, Removed malformed {malformed_counter} nan {nan_counter}")
    print(labelled_dataframe.groupby("labelled_device").agg(packets=("labelled_device", "size"), sources=("source", "nunique")).sort_values(by="packets", ascending=False))
    labelled_dataframe = assign_majority_label(labelled_dataframe)

    # CODE FIELD 3 inline
    print("\\n--- After majority vote ---")
    print(labelled_dataframe.groupby("labelled_device").agg(packets=("labelled_device", "size"), sources=("source", "nunique")).sort_values(by="packets", ascending=False))
    print(labelled_dataframe.groupby("group_id").agg(packets=("group_id", "size"), sources=("source", "nunique")).sort_values(by="group_id"))
    print(labelled_dataframe.groupby(["labelled_device","group_id"]).agg(packets=("labelled_device", "size"), sources=("source", "nunique")))

    # save
    new_file_name = pathlib.Path(fname).stem + "_labeled"
    labelled_dataframe.to_csv(base / f"{new_file_name}.csv", index=False)
    print(f"WROTE {base / (new_file_name+'.csv')}")


['2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours0To2.csv', '2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours2To4.csv', '2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours4To6.csv', '2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours6To8.csv', '2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours8To10.csv']
\n=== 2Groups_3Devices_10h_Seewis_IphoneAirpodsSamsungTag_DiffDistances_Hours0To2.csv ===
target_group_rsp_scrs: ['44:55:3c:29:d4:d8', '48:29:85:2c:a6:87', '4d:39:86:43:78:78', '55:38:ef:cf:40:14', '5a:9e:c1:f9:4f:81', '5e:61:60:ab:23:36', '6b:eb:0f:72:4d:54', '6c:38:7c:46:cc:33', '6c:d9:d6:e7:5a:20', '43:65:e1:d6:f2:eb', '4a:9b:a5:44:83:28', '51:dc:9f:2e:24:8c', '5e:11:30:21:6c:39', '65:53:f9:ec:76:2e', '6c:8a:13:1a:55:f5', '6d:d3:53:77:f9:6f', '7b:0c:20:7c:f3:7e', '7d:89:59:c3:7e:eb']
Only Malformed Sources: 20559
Empty Sources: 10825
Sources total: 31535, labelled rows: 729

In [10]:
def group_by_src(df):
    # Simple robust version - converts RSSI to numeric first
    df = df.copy()
    df["rssi_num"] = pd.to_numeric(df["rssi"].astype(str).str.replace("dBm","",regex=False).str.strip(), errors="coerce")
    # Use simple agg without dict lambda that can fail
    result = (
    df.groupby("source")
      .agg(
          count=("rssi_num", "count"),
          RSSI_min=("rssi_num", "min"),
          RSSI_max=("rssi_num", "max"),
          RSSI_avg=("rssi_num", "mean"),
          label=("labelled_device", lambda x: ", ".join(x.unique())),
          group=("group_id", lambda x: ", ".join(map(str, x.unique())))
      )
      .reset_index()                
      .sort_values(by="count", ascending=False)
      .reset_index(drop=True)
    )
    return result
